# DeepVoice: Do Fake-Voice Detectors Learn Fakeness, or Just the Dataset?

**CMPS 261 Introduction to Machine Learning, AUB, Fall 2026.** Hashem Awad, Ahmed Faleh, Ali Sleiman.
Report: submitted on Moodle. Code: https://github.com/HashemAwad/deepvoice

**Question.** What do fake-voice detectors learn, and how much of their performance survives when the
data change? We train a ladder of detectors on ASVspoof 2019 LA (rung 0: silence/duration only;
rung 1: tuned classical models on LFCC; rung 2: frozen XLS-R-300M + logistic regression), test them on
unseen attacks and on In-the-Wild, and probe shortcuts, calibration and failures.

**Rules that every cell follows** (see `CLAUDE.md`): fit on train only; choose settings and thresholds
on dev; eval and In-the-Wild were used only after `hypotheses.md` was frozen and hashed (commit
`1d896cd`, sha256 in the README); primary metric EER with a speaker-clustered bootstrap 95% CI;
features computed once and cached; seed 0 and a config file for every run.

**How to run.** `./scripts/setup.sh --dl`, `./scripts/download_data.sh` (about 16 GB), then
`uv run --extra dl python -m deepvoice.pipeline` (builds all caches; about 3 h from scratch, seconds
when cached), then run this notebook top to bottom. Each section below is one of the repository's
notebooks 00-06, unchanged; reusable code lives in `src/deepvoice/`.

## 0. Build (or load) every cached feature set

In [ ]:
from pathlib import Path

from deepvoice.pipeline import build_all

build_all("../data" if Path.cwd().name == "notebooks" else "data")

---
# 1. Data audit

Verified split counts, class balance, duration and silence by class (train and dev). Figure 1 of the report. *(Source: `notebooks/00_audit.ipynb`.)*

## 00 Data audit (ASVspoof 2019 LA, train + dev only)

Class balance, duration and leading/trailing silence by class. Eval and In-the-Wild are not
touched here (they stay locked until `hypotheses.md` is frozen). Silence comes from the
energy VAD in `deepvoice.silence` (frame = speech if within 35 dB of the clip's loudest frame).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
proto = pd.read_parquet(ROOT / "data/protocols.parquet")
df = pd.concat(
    [pd.read_parquet(ROOT / f"data/features/silence_{s}.parquet").assign(split=s) for s in ["train", "dev"]]
).merge(proto[["file", "label", "attack", "speaker"]], on="file")
df["cls"] = df["label"].map({0: "real", 1: "fake"})

## Protocol counts (all splits; labels only, no audio)

In [ ]:
from deepvoice.data import summarize

counts = summarize(proto)
counts.to_csv(ROOT / "results/protocol_counts.csv", index=False)
counts

## Class balance

In [ ]:
bal = df.groupby(["split", "cls"], sort=False).size().unstack()
bal["fake_share"] = (bal["fake"] / bal.sum(axis=1)).round(3)
bal

## Duration and silence by class (median and interquartile range)

In [ ]:
rows = []
for (split, cls), g in df.groupby(["split", "cls"], sort=False):
    r = {"split": split, "class": cls, "n": len(g)}
    for c in ["duration", "lead_sil", "trail_sil", "sil_ratio", "energy_max"]:
        r[c + "_median"] = round(g[c].median(), 3)
        r[c + "_iqr"] = f"{g[c].quantile(.25):.2f}-{g[c].quantile(.75):.2f}"
    rows.append(r)
audit = pd.DataFrame(rows)
audit.to_csv(ROOT / "results/audit_by_class.csv", index=False)
audit.T

In [ ]:
df[df.split == "train"].groupby("attack")[["duration", "lead_sil", "trail_sil", "sil_ratio"]].median().round(3)

## Figure for the progress report

In [ ]:
REAL, FAKE = "#2a78d6", "#eb6834"  # categorical slots 1-2 (colour-blind checked)
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(1, 3, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1, 2.4, 1.3]})

# (a) class balance
for i, split in enumerate(["train", "dev"]):
    r, f = bal.loc[split, "real"], bal.loc[split, "fake"]
    ax[0].barh(i, r, color=REAL, height=0.6)
    ax[0].barh(i, f, left=r + 300, color=FAKE, height=0.6)
    ax[0].text(r + f + 900, i, f"{f / (r + f):.0%} fake", va="center", color=INK)
ax[0].set_yticks([0, 1], ["train", "dev"]); ax[0].invert_yaxis()
ax[0].set_xlim(0, 33000); ax[0].set_xlabel("clips")
ax[0].set_title("(a) Class balance", loc="left", color=INK)

# (b) leading silence: real vs each training attack, train and dev side by side
groups = ["bonafide", "A01", "A02", "A03", "A04", "A05", "A06"]
for j, split in enumerate(["train", "dev"]):
    d = df[df.split == split]
    data = [d.loc[d.attack == g, "lead_sil"] for g in groups]
    pos = [k * 3 + j for k in range(len(groups))]
    bp = ax[1].boxplot(data, positions=pos, widths=0.8, showfliers=False, patch_artist=True,
                       medianprops={"color": INK, "linewidth": 1.2})
    for patch, g in zip(bp["boxes"], groups):
        patch.set(facecolor=REAL if g == "bonafide" else FAKE, edgecolor="white",
                  alpha=1.0 if split == "train" else 0.55)
ax[1].set_xticks([k * 3 + 0.5 for k in range(len(groups))], ["real"] + groups[1:])
ax[1].set_ylabel("leading silence (s)"); ax[1].grid(axis="y", color=GRID, linewidth=0.6)
ax[1].set_title("(b) Leading silence by attack (left box train, right box dev)", loc="left", color=INK)

# (c) duration by class
for cls, col in [("real", REAL), ("fake", FAKE)]:
    ax[2].hist(df.loc[df.cls == cls, "duration"], bins=60, range=(0, 10), density=True,
               histtype="step", linewidth=2, color=col, label=cls)
ax[2].set_xlabel("duration (s)"); ax[2].set_yticks([])
ax[2].legend(frameon=False); ax[2].set_title("(c) Clip duration (train+dev)", loc="left", color=INK)

fig.tight_layout()
(ROOT / "results/figures").mkdir(exist_ok=True)
fig.savefig(ROOT / "results/figures/audit_silence.png", dpi=200)
fig.savefig(ROOT / "results/figures/audit_silence.pdf")

---
# 2. E2: feature engineering

Feature-group, MFCC-vs-LFCC, coefficient, delta and pooling ablations with fixed untuned models (dev only). *(Source: `notebooks/01_features.ipynb`.)*

## 01 E2: feature engineering ablation (train -> dev)

Fixed, untuned models: logistic regression (C=1) and random forest (300 trees), seed 0. Every
setting is trained on train and scored on **dev** with a speaker-clustered bootstrap 95% CI.
Eval is not used here. Results: `results/e2.csv`; figure: `results/figures/e2_features.png`.

Groups: S = silence/duration/energy (9), M = MFCC, L = LFCC, P = spectral (6 descriptors).
Unless stated otherwise M and L use 20 coefficients + delta + delta-delta, pooled mean+std.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.features import load_xy
from deepvoice.models import run

MODELS = {"logreg": {"C": 1.0}, "rf": {"n_estimators": 300, "n_jobs": 6}}
NAMES = {"silence": "S", "mfcc": "M", "lfcc": "L", "spec": "P"}
MS, MSMM, MSPCT = ("mean", "std"), ("mean", "std", "min", "max"), ("mean", "std", "p10", "p50", "p90")


def label(groups, n_coef=20, deltas=2, stats=MS):
    if len(groups) == 1 and groups[0] in ("mfcc", "lfcc"):
        pool = {MS: "meanstd", MSMM: "meanstd_minmax", MSPCT: "meanstd_pct"}[stats]
        return f"{groups[0]}{n_coef}_d{deltas}_{pool}"
    return "+".join(NAMES[g] for g in groups)


settings = {}
def add(family, groups, **kw):
    settings.setdefault(label(groups, **kw), (family, groups, kw))

ALL = ["silence", "mfcc", "lfcc", "spec"]
for g in ALL:
    add("group: one", [g])
for g in ALL:
    add("group: drop one", [x for x in ALL if x != g])
add("group: all", ALL)
for g in ["mfcc", "lfcc"]:
    for n in [13, 20, 40]:
        for d in [0, 1, 2]:
            add("coefficients x deltas", [g], n_coef=n, deltas=d)
    for st in [MSMM, MSPCT]:
        add("pooling", [g], stats=st)
print(len(settings), "feature settings x", len(MODELS), "models")

In [ ]:
RERUN = False  # True recomputes settings already in results/e2.csv
out = ROOT / "results/e2.csv"
done = set(pd.read_csv(out)["config"]) if out.exists() and not RERUN else set()
for fs, (family, groups, kw) in settings.items():
    todo = [m for m in MODELS if f"e2_{m}_{fs}_s0" not in done]
    if not todo:
        continue
    tr, dv = load_xy("train", groups, **kw), load_xy("dev", groups, **kw)
    X = list(tr.columns[4:])
    for m in todo:
        run(1, m, X, tr, [("asvspoof", "dev", dv)], params=MODELS[m], experiment="e2", feature_set=fs)
    print(f"{fs:28s} {len(X):4d} features done")

## Results (dev EER %, 95% CI)

In [ ]:
r = pd.read_csv(out)
r["setting"] = r["model"].str.split(":").str[1]
r["model"] = r["model"].str.split(":").str[0]
r["family"] = r["setting"].map(lambda s: settings[s][0])
r["n_features"] = r["config"].map(lambda c: len(pd.read_json(ROOT / f"results/configs/{c}.json", typ="series")["features"]))
r["dev EER %"] = r.apply(lambda x: f"{100*x.eer:.2f} [{100*x.ci_low:.2f}, {100*x.ci_high:.2f}]", axis=1)
table = r.pivot_table(index=["family", "setting", "n_features"], columns="model", values="dev EER %", aggfunc="first")
table.to_csv(ROOT / "results/e2_table.csv")
table

## Figure

In [ ]:
LR, RF = "#1baf7a", "#6250d6"  # model colours (validated palette); real/fake keep blue/orange
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})

# (a) feature groups: alone, all, all-but-one
order = ["S", "P", "mfcc20_d2_meanstd", "lfcc20_d2_meanstd", "S+M+L+P",
         "M+L+P", "S+L+P", "S+M+P", "S+M+L"]
names = ["S only", "P only", "M only", "L only", "all (S+M+L+P)",
         "all - S", "all - M", "all - L", "all - P"]
for k, (m, col) in enumerate([("logreg", LR), ("rf", RF)]):
    d = r[r.model == m].set_index("setting").loc[order]
    y = [i + (k - 0.5) * 0.38 for i in range(len(order))]
    ax[0].barh(y, 100 * d.eer, height=0.36, color=col, label={"logreg": "logistic regression", "rf": "random forest"}[m])
    ax[0].errorbar(100 * d.eer, y, xerr=[100 * (d.eer - d.ci_low), 100 * (d.ci_high - d.eer)],
                   fmt="none", ecolor=INK, elinewidth=0.8, capsize=2)
ax[0].set_yticks(range(len(order)), names); ax[0].invert_yaxis()
ax[0].set_xscale("log"); ax[0].set_xlabel("dev EER % (log scale; bars = 95% CI)")
ax[0].grid(axis="x", color=GRID, linewidth=0.6); ax[0].legend(frameon=False, loc="lower right")
ax[0].set_title("(a) Feature groups", loc="left", color=INK)

# (b) MFCC vs LFCC by number of coefficients (with delta + delta-delta)
for m, col in [("logreg", LR), ("rf", RF)]:
    for g, ls, mk in [("mfcc", "--", "o"), ("lfcc", "-", "s")]:
        d = r[(r.model == m)].set_index("setting").loc[[f"{g}{n}_d2_meanstd" for n in [13, 20, 40]]]
        ax[1].plot([13, 20, 40], 100 * d.eer, ls=ls, marker=mk, ms=6, lw=2, color=col)
        ax[1].text(41.5, 100 * d.eer.iloc[-1], f"{g.upper()} {'LR' if m == 'logreg' else 'RF'}",
                   va="center", color=INK, fontsize=8)
ax[1].set_yscale("log"); ax[1].set_xticks([13, 20, 40]); ax[1].set_xlim(11, 48)
ax[1].set_xlabel("number of coefficients (+ delta + delta-delta)"); ax[1].set_ylabel("dev EER % (log scale)")
ax[1].grid(axis="y", color=GRID, linewidth=0.6)
ax[1].set_title("(b) MFCC (dashed) vs LFCC (solid)", loc="left", color=INK)
from matplotlib.ticker import FuncFormatter, NullFormatter

for a, axis in [(ax[0], "x"), (ax[1], "y")]:
    getattr(a, f"{axis}axis").set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    getattr(a, f"{axis}axis").set_minor_formatter(NullFormatter())
fig.tight_layout()
fig.savefig(ROOT / "results/figures/e2_features.png", dpi=200)
fig.savefig(ROOT / "results/figures/e2_features.pdf")

---
# 3. E3: tuning, overfitting and split sensitivity

Random search on LFCC-40, selection on dev vs speaker-grouped CV, gap table and learning curves. *(Source: `notebooks/02_baselines.ipynb`.)*

## 02 E3: tuning, overfitting and split sensitivity (train -> dev)

Fixed features: LFCC, 40 coefficients + delta + delta-delta, mean+std (240 columns; best LFCC setting in E2). Seed 0.
Models: logistic regression, random forest, HistGradientBoosting (all class-balanced).

1. **Random search** (30 settings per model). Each setting is scored on dev and by
   speaker-grouped 5-fold CV on train. Full table: `results/e3_search.csv`.
2. **Selection** by dev and, separately, by grouped CV (split sensitivity), plus untuned defaults.
3. **Gap table**: train vs grouped-CV vs dev EER (eval is added later, once per final model).
4. **Learning curves**: dev-selected settings at 5/10/25/50/100% of train.

Final rows with 95% CIs: `results/e3.csv`. Figure: `results/figures/e3_tuning.png`.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import loguniform

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.features import load_xy
from deepvoice.models import random_search, run

FS = {"groups": ("lfcc",), "n_coef": 40, "deltas": 2, "stats": ("mean", "std")}
train, dev = load_xy("train", **FS), load_xy("dev", **FS)
X = list(train.columns[4:])
print(train.shape, dev.shape, len(X), "features")

## 1. Random search

In [ ]:
SPACES = {
    "logreg": ({"C": loguniform(1e-4, 1e2), "penalty": ["l1", "l2"]}, {"solver": "liblinear"}),
    "rf": ({"n_estimators": [100, 200, 400], "max_depth": [None, 10, 20, 30],
            "min_samples_leaf": [1, 2, 5, 10], "max_features": ["sqrt", "log2", 0.3]}, {"n_jobs": 6}),
    "hgb": ({"learning_rate": loguniform(0.02, 0.3), "max_depth": [None, 3, 5, 8],
             "l2_regularization": loguniform(1e-3, 10), "max_iter": [100, 200, 400]},
            {"early_stopping": False}),
}
DEFAULTS = {"logreg": {"C": 1.0}, "rf": {"n_estimators": 300, "n_jobs": 6}, "hgb": {}}

search_csv = ROOT / "results/e3_search.csv"
if search_csv.exists():  # computed once; delete the file on purpose to redo the search
    search = pd.read_csv(search_csv)
else:
    search = pd.concat([random_search(m, space, train, dev, X, n_iter=30, seed=0, fixed=fixed)
                        for m, (space, fixed) in SPACES.items()], ignore_index=True)
    search.to_csv(search_csv, index=False, float_format="%.6g")
search.groupby("model")[["dev_eer", "cv_eer", "fit_time"]].describe().round(4).T

## 2. Selection on dev vs on grouped CV, plus untuned defaults

In [ ]:
picks = {}
for m in SPACES:
    s = search[search.model == m]
    picks[(m, "default")] = DEFAULTS[m]
    picks[(m, "dev")] = json.loads(s.loc[s.dev_eer.idxmin(), "params"])
    picks[(m, "cv")] = json.loads(s.loc[s.cv_eer.idxmin(), "params"])
    print(f"{m:7s} same setting chosen by dev and CV: {picks[(m, 'dev')] == picks[(m, 'cv')]}")

for (m, sel), params in picks.items():
    run(1, m, X, train, [("asvspoof", "train", train), ("asvspoof", "dev", dev)],
        params=params, experiment="e3", feature_set=f"lfcc40dd_{sel}sel" if sel != "default" else "lfcc40dd_default")

## 3. Gap table: train vs speaker-grouped CV vs dev (EER %)

In [ ]:
from deepvoice.models import grouped_cv_eer

r = pd.read_csv(ROOT / "results/e3.csv")
rows = []
for (m, sel), params in picks.items():
    fs = f"lfcc40dd_{sel}sel" if sel != "default" else "lfcc40dd_default"
    rr = r[r.config == f"e3_{m}_{fs}_s0"].set_index("split")
    if sel == "default":
        cv = grouped_cv_eer(m, params, train, X)
    else:
        s = search[(search.model == m) & (search.params == json.dumps(params))]
        cv = s.cv_eer.iloc[0]
    rows.append({"model": m, "selected by": sel, "train": 100 * rr.loc["train", "eer"], "grouped CV": 100 * cv,
                 "dev": 100 * rr.loc["dev", "eer"],
                 "dev 95% CI": f"{100*rr.loc['dev','ci_low']:.2f}-{100*rr.loc['dev','ci_high']:.2f}",
                 "fit s": rr.loc["dev", "train_time"], "params": json.dumps(params)})
gap = pd.DataFrame(rows)
gap.to_csv(ROOT / "results/e3_gap.csv", index=False, float_format="%.4g")
gap.round(3)

## 4. Learning curves (dev-selected settings)

In [ ]:
FRACS = [0.05, 0.10, 0.25, 0.50, 1.00]
for m in SPACES:
    for f in FRACS:
        sub = train if f == 1 else train.groupby("label", group_keys=False).sample(frac=f, random_state=0)
        run(1, m, X, sub, [("asvspoof", "dev", dev)], params=picks[(m, "dev")],
            experiment="e3", feature_set=f"lfcc40dd_devsel_frac{int(f * 100):03d}")
lc = pd.read_csv(ROOT / "results/e3.csv")
lc = lc[lc.config.str.contains("_frac")].copy()
lc["frac"] = lc.config.str.extract(r"frac(\d+)").astype(int)[0]
lc["model"] = lc.model.str.split(":").str[0]
lc.pivot_table(index="frac", columns="model", values="eer").mul(100).round(2)

## Figure

In [ ]:
COL = {"logreg": "#1baf7a", "rf": "#6250d6", "hgb": "#eda100"}
NAME = {"logreg": "logistic regression", "rf": "random forest", "hgb": "gradient boosting"}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# (a) learning curves with CI band
for m in SPACES:
    d = lc[lc.model == m].sort_values("frac")
    ax[0].plot(d.frac, 100 * d.eer, marker="o", ms=6, lw=2, color=COL[m], label=NAME[m])
    ax[0].fill_between(d.frac, 100 * d.ci_low, 100 * d.ci_high, color=COL[m], alpha=0.15, lw=0)
ax[0].set_xscale("log"); ax[0].set_xticks([5, 10, 25, 50, 100], ["5%", "10%", "25%", "50%", "100%"])
ax[0].set_xlabel("share of train used"); ax[0].set_ylabel("dev EER % (band = 95% CI)")
ax[0].grid(axis="y", color=GRID, linewidth=0.6); ax[0].legend(frameon=False)
ax[0].set_title("(a) Learning curves (dev-selected settings)", loc="left", color=INK)

# (b) gap: train / grouped CV / dev for the dev-selected models
g = gap[gap["selected by"] == "dev"].set_index("model")
marks = [("train", "o", "train"), ("grouped CV", "s", "speaker-grouped CV"), ("dev", "D", "dev")]
for i, m in enumerate(SPACES):
    vals = [g.loc[m, c] for c, _, _ in marks]
    ax[1].plot([min(vals), max(vals)], [i, i], color=COL[m], lw=2)
    for (c, mk, _), v in zip(marks, vals):
        ax[1].plot(v, i, marker=mk, ms=8, color=COL[m], mec="white", mew=1.5, ls="none")
for c, mk, lab in marks:
    ax[1].plot([], [], marker=mk, ms=7, color=MUTED, ls="none", label=lab)
ax[1].set_yticks(range(len(SPACES)), [NAME[m] for m in SPACES]); ax[1].invert_yaxis()
ax[1].set_xlabel("EER %"); ax[1].grid(axis="x", color=GRID, linewidth=0.6)
ax[1].legend(frameon=False, loc="lower right")
ax[1].set_title("(b) Train vs grouped CV vs dev (tuned on dev)", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/e3_tuning.png", dpi=200)
fig.savefig(ROOT / "results/figures/e3_tuning.pdf")

---
# 4. Rung 2: frozen XLS-R-300M

Layer sweep on dev, small MLP, the pre-registered selection rule, and the single eval run of the chosen model. *(Source: `notebooks/03_xlsr.ipynb`.)*

## 03 Rung 2: frozen XLS-R-300M, layer sweep (train -> dev)

Embeddings come from `deepvoice.xlsr.extract_cached` (mean-pooled every 3rd layer, fp16, cached in
`data/features/`). For each layer: standardized logistic regression (C=1, class-balanced, seed 0),
trained on train and scored on dev with a speaker-clustered 95% CI. Results: `results/rung2.csv`.
The layer for rung 2 is chosen on dev.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.models import run
from deepvoice.xlsr import LAYERS

p = pd.read_parquet("data/protocols.parquet")
COLS = [f"x{i:04d}" for i in range(1024)]
D = {}
for s in ["train", "dev"]:
    m = p[p.split == s][["file", "label", "speaker"]].reset_index(drop=True)
    assert Path(f"data/features/xlsr300m_{s}_files.txt").read_text().split() == list(m.file)
    D[s] = (m, np.load(f"data/features/xlsr300m_{s}.npy", mmap_mode="r"))

In [ ]:
out = ROOT / "results/rung2.csv"
done = set(pd.read_csv(out).config) if out.exists() else set()
for k, layer in enumerate(LAYERS):
    if f"rung2_logreg_xlsr300m_L{layer:02d}_s0" in done:
        continue
    fr = {s: pd.concat([m, pd.DataFrame(np.asarray(e[:, k], dtype=np.float32), columns=COLS)], axis=1)
          for s, (m, e) in D.items()}
    run(2, "logreg", COLS, fr["train"], [("asvspoof", "train", fr["train"]), ("asvspoof", "dev", fr["dev"])],
        params={"C": 1.0}, feature_set=f"xlsr300m_L{layer:02d}")
r = pd.read_csv(out)
r["layer"] = r.model.str.extract(r"L(\d+)").astype(int)[0]
dev = r[r.split == "dev"].sort_values("layer")
dev[["layer", "eer", "ci_low", "ci_high", "train_time"]].assign(**{c: dev[c] * 100 for c in ["eer", "ci_low", "ci_high"]}).round(3)

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(figsize=(5.2, 3.2))
ax.fill_between(dev.layer, 100 * dev.ci_low, 100 * dev.ci_high, color="#1baf7a", alpha=0.18, lw=0)
ax.plot(dev.layer, 100 * dev.eer, marker="o", ms=6, lw=2, color="#1baf7a", label="XLS-R layer + logistic regression")
ax.axhline(0.49, color=MUTED, ls="--", lw=1)
ax.text(24.3, 0.49, "best classical\n(LFCC40 RF, untuned)", va="center", fontsize=8, color=MUTED)
ax.set_xticks(LAYERS); ax.set_xlim(-0.5, 30)
ax.set_xlabel("XLS-R layer (0 = CNN output, 24 = last)"); ax.set_ylabel("dev EER % (band = 95% CI)")
ax.grid(axis="y", color=GRID, linewidth=0.6)
ax.set_title("Frozen XLS-R-300M: dev EER by layer", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/xlsr_layers.png", dpi=200)
fig.savefig(ROOT / "results/figures/xlsr_layers.pdf")

## Small MLP on the best layers, and the rung 2 choice (decided before eval)

Rule, fixed before any eval run: take the rung 2 candidate with the lowest dev EER; every candidate
whose dev EER lies inside that model's 95% CI counts as tied, and ties are broken by speaker-grouped
5-fold CV EER on train; if CV ties too, the lower dev EER wins.

*Note (20 Nov):* an earlier version broke an exact CV tie by row order; in one re-run that picked
layer 6 and scored it on eval once (0.34% EER, kept in `results/rung2.csv` for transparency). The
final model stays layer 3, the lowest-dev-EER model as `hypotheses.md` defines it. Candidates: logistic regression on each layer, and an MLP (one hidden layer of
256, early stopping) on the tied layers.

In [ ]:
from deepvoice.models import grouped_cv_eer


def frames(layer, splits):
    k = LAYERS.index(layer)
    return {s: pd.concat([D[s][0], pd.DataFrame(np.asarray(D[s][1][:, k], dtype=np.float32), columns=COLS)], axis=1)
            for s in splits}

r = pd.read_csv(out); dev = r[r.split == "dev"].copy()
best = dev.loc[dev.eer.idxmin()]
tied_layers = sorted(int(m.split("L")[1]) for m in dev[dev.eer <= best.ci_high].model)
print("best:", best.model, f"{100 * best.eer:.4f}%", "| tied layers:", tied_layers)
MLP = {"hidden_layer_sizes": (256,), "alpha": 1e-3}
for layer in tied_layers:
    if f"rung2_mlp_xlsr300m_L{layer:02d}_s0" not in set(r.config):
        fr = frames(layer, ["train", "dev"])
        run(2, "mlp", COLS, fr["train"], [("asvspoof", "train", fr["train"]), ("asvspoof", "dev", fr["dev"])],
            params=MLP, feature_set=f"xlsr300m_L{layer:02d}")
r = pd.read_csv(out); dev = r[r.split == "dev"].copy()
best = dev.loc[dev.eer.idxmin()]
tied = dev[dev.eer <= best.ci_high].copy()
for i, row in tied.iterrows():
    name, layer = row.model.split(":")[0], int(row.model.split("L")[1])
    params = {"C": 1.0} if name == "logreg" else MLP
    tied.loc[i, "cv_eer"] = grouped_cv_eer(name, params, frames(layer, ["train"])["train"], COLS)
# ties on CV fall back to the lower dev EER, so the choice never depends on row order
choice = tied.sort_values(["cv_eer", "eer"]).iloc[0]
tied[["model", "eer", "ci_low", "ci_high", "cv_eer"]].assign(**{c: tied[c] * 100 for c in ["eer", "ci_low", "ci_high", "cv_eer"]}).round(4)

In [ ]:
CHOICE = choice.model
print("rung 2 model (chosen on dev + grouped CV, before eval):", CHOICE)

## Rung 2 on ASVspoof eval (scored once)

In [ ]:
from deepvoice.data import REPEATED_ATTACKS
from deepvoice.metrics import per_attack_eer

name, layer = CHOICE.split(":")[0], int(CHOICE.split("L")[1])
params = {"C": 1.0} if name == "logreg" else MLP
m = p[p.split == "eval"][["file", "label", "speaker", "attack"]].reset_index(drop=True)
assert Path("data/features/xlsr300m_eval_files.txt").read_text().split() == list(m.file)
ev = np.load("data/features/xlsr300m_eval.npy", mmap_mode="r")
D["eval"] = (m[["file", "label", "speaker"]], ev)
fr = frames(layer, ["train", "dev", "eval"])
model, rows = run(2, name, COLS, fr["train"], [("asvspoof", s, fr[s]) for s in ["train", "dev", "eval"]],
                  params=params, feature_set=f"xlsr300m_L{layer:02d}")
s = model.predict_proba(fr["eval"][COLS].to_numpy())[:, 1]
pa = pd.Series(per_attack_eer(m.label, s, m.attack)).mul(100)
pa = pd.DataFrame({CHOICE: pa, "type": ["repeated" if a in REPEATED_ATTACKS else "unseen" for a in pa.index]})
pa.to_csv(ROOT / "results/rung2_eval_per_attack.csv", float_format="%.2f")
print({r_["split"]: f"{100*r_['eer']:.2f}% [{100*r_['ci_low']:.2f}, {100*r_['ci_high']:.2f}]" for r_ in rows})
pa.round(2).T

---
# 5. E4: generalization

Rung 0 per attack, then all three final models refit from code and scored on dev, eval and In-the-Wild; H3 and H4 verdicts. *(Source: `notebooks/04_generalization.ipynb`.)*

## 04 Generalization: per-attack EER on ASVspoof eval

Run only after `hypotheses.md` was frozen (commit 1d896cd). Each final model is scored on eval once.
Per-attack EER = that attack's fakes against all eval real clips (`deepvoice.metrics.per_attack_eer`).
A16 and A19 reuse training algorithms A04 and A06 ("repeated"); the other 11 are unseen.

**Rung 0** (silence/duration only). The rung 0 model is the one with the lower dev EER: the
depth-3 tree (dev 28.0% vs 30.4% for logistic regression). It is refit here deterministically
(seed 0, identical to `results/rung0.csv`).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.data import REPEATED_ATTACKS
from deepvoice.metrics import per_attack_eer
from deepvoice.models import build

p = pd.read_parquet("data/protocols.parquet")
S = {s: pd.read_parquet(f"data/features/silence_{s}.parquet").merge(p[["file", "label", "speaker", "attack"]], on="file")
     for s in ["train", "eval"]}
F = ["lead_sil", "trail_sil", "sil_ratio", "duration"]
rows = {}
for m, params in [("tree", {"max_depth": 3}), ("logreg", {"C": 1.0})]:
    model = build(m, 0, **params).fit(S["train"][F].to_numpy(), S["train"].label.to_numpy())
    s = model.predict_proba(S["eval"][F].to_numpy())[:, 1]
    rows[m] = per_attack_eer(S["eval"].label, s, S["eval"].attack)
pa = pd.DataFrame(rows).mul(100)
pa["type"] = ["repeated" if a in REPEATED_ATTACKS else "unseen" for a in pa.index]
pa.to_csv(ROOT / "results/rung0_eval_per_attack.csv", float_format="%.2f")
r0 = pd.read_csv(ROOT / "results/rung0.csv").query("split == 'eval'").set_index("model")
pa.round(1)

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(figsize=(6.4, 3.2))
d = pa["tree"]
x = range(len(d))
bars = ax.bar(x, d, color=["#c3c2b7" if t == "repeated" else "#52514e" for t in pa.type], width=0.7)
for b, t in zip(bars, pa.type):
    if t == "repeated":
        b.set_hatch("////"); b.set_edgecolor("white")
for i, v in zip(x, d):
    ax.text(i, v + 1, f"{v:.0f}", ha="center", fontsize=7.5, color=INK)
ax.axhline(50, color=MUTED, ls=":", lw=1); ax.text(-0.4, 51, "chance (50%)", fontsize=8, color=MUTED)
ax.axhline(100 * r0.loc["tree", "eer"], color="#e34948", ls="--", lw=1.2)
ax.text(-0.4, 100 * r0.loc["tree", "eer"] + 1.5, f"pooled eval EER {100 * r0.loc['tree', 'eer']:.1f}%", fontsize=8, color="#e34948")
ax.set_xticks(list(x), d.index); ax.set_ylim(0, 60)
ax.set_ylabel("eval EER % (silence/duration only)")
from matplotlib.patches import Patch

ax.legend(handles=[Patch(color="#52514e", label="unseen attack"), Patch(facecolor="#c3c2b7", hatch="////", edgecolor="white", label="repeated (A16=A04, A19=A06)")],
          frameon=False, loc="upper left", bbox_to_anchor=(0, 0.72), fontsize=8)
ax.set_title("Rung 0 on ASVspoof eval, per attack", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/rung0_eval_per_attack.png", dpi=200)
fig.savefig(ROOT / "results/figures/rung0_eval_per_attack.pdf")

## E4: the three final models on In-the-Wild (H3, H4)

The final model of each rung (chosen on dev, see notebooks 02/03 and `hypotheses.md`) is refit here
from code with its saved settings (deterministic, seed 0) and scored on train, dev, eval and
In-the-Wild (ITW). The eval numbers must equal the ones already recorded; this re-runs the same
models, it is not a new look at eval.

- **H3 (ITW part):** the rung 2 minus rung 1 margin is reported on eval and ITW; a shrinkage is
  only claimed if it is >= 10 pp.
- **H4:** on ITW, error at the dev threshold (mean of false-accept and false-reject rate) minus the
  best-threshold error (ITW EER) >= 10 pp for at least 2 of the 3 rungs.
- Rung 0 on ITW is the shortcut control: if silence alone separates ITW, ITW has a shortcut too.

In [ ]:
import json

import numpy as np

from deepvoice.features import load_xy
from deepvoice.metrics import eer, error_at_threshold
from deepvoice.models import run
from deepvoice.xlsr import LAYERS

SPLITS = ["train", "dev", "eval", "itw"]
DATASET = {"train": "asvspoof", "dev": "asvspoof", "eval": "asvspoof", "itw": "itw"}


def xlsr_frames(layer):
    k, cols, out = LAYERS.index(layer), [f"x{i:04d}" for i in range(1024)], {}
    for s in SPLITS:
        meta = load_xy(s, groups=())
        assert Path(f"data/features/xlsr300m_{s}_files.txt").read_text().split() == list(meta.file)
        e = np.load(f"data/features/xlsr300m_{s}.npy", mmap_mode="r")
        out[s] = pd.concat([meta, pd.DataFrame(np.asarray(e[:, k], dtype=np.float32), columns=cols)], axis=1)
    return out, cols


gap = pd.read_csv(ROOT / "results/e3_gap.csv")
hgb_params = json.loads(gap[(gap.model == "hgb") & (gap["selected by"] == "dev")].params.iloc[0])
lfcc = {s: load_xy(s, groups=("lfcc",), n_coef=40, deltas=2, stats=("mean", "std")) for s in SPLITS}
sil = {s: load_xy(s, groups=("silence",)) for s in SPLITS}
xl, xcols = xlsr_frames(3)
FINAL = [  # rung, model, data, feature columns, params, experiment, feature_set
    (0, "tree", sil, ["lead_sil", "trail_sil", "sil_ratio", "duration"], {"max_depth": 3}, "rung0", None),
    (1, "hgb", lfcc, list(lfcc["train"].columns[4:]), hgb_params, "rung1", "lfcc40dd_e3devsel"),
    (2, "logreg", xl, xcols, {"C": 1.0}, "rung2", "xlsr300m_L03"),
]

In [ ]:
rows, scores = [], {}
for rung, name, D, X, params, exp, fs in FINAL:
    config = f"{exp}_{name}_{fs}_s0" if fs else f"{exp}_{name}_s0"
    before = pd.read_csv(ROOT / f"results/{exp}.csv").query("config == @config and split == 'eval'").eer
    model, res = run(rung, name, X, D["train"], [(DATASET[s], s, D[s]) for s in SPLITS],
                     params=params, experiment=exp, feature_set=fs)
    after = {r["split"]: r for r in res}  # results CSVs keep 6 significant digits
    assert len(before) == 1 and np.isclose(before.iloc[0], after["eval"]["eer"], rtol=1e-5, atol=0), "eval not reproduced"
    sc = {s: model.predict_proba(D[s][X].to_numpy())[:, 1] for s in SPLITS}
    scores[rung] = sc
    thr = eer(D["dev"].label, sc["dev"])[1]
    row = {"rung": rung, "model": f"{name}:{fs}" if fs else name, "dev_threshold": thr}
    for s in ["dev", "eval", "itw"]:
        row[f"{s}_eer"], row[f"{s}_ci"] = after[s]["eer"], f"{100*after[s]['ci_low']:.1f}-{100*after[s]['ci_high']:.1f}"
        row[f"{s}_err_at_dev_thr"] = error_at_threshold(D[s].label, sc[s], thr)
    rows.append(row)
e4 = pd.DataFrame(rows)
e4.to_csv(ROOT / "results/e4_itw.csv", index=False, float_format="%.6g")
show = e4.copy()
for c in [c for c in e4.columns if c.endswith("_eer") or c.endswith("_thr") and c != "dev_threshold"]:
    show[c] = (100 * show[c]).round(2)
show

## Hypothesis verdicts from E4 (rules from the frozen `hypotheses.md`)

In [ ]:
by = e4.set_index("rung")
m_eval = by.loc[1, "eval_eer"] - by.loc[2, "eval_eer"]
m_itw = by.loc[1, "itw_eer"] - by.loc[2, "itw_eer"]
shrink = m_eval - m_itw
h4_gaps = by["itw_err_at_dev_thr"] - by["itw_eer"]
verdicts = pd.DataFrame([
    {"hypothesis": "H3 (ITW part)",
     "evidence": f"rung2-rung1 margin: eval {100*m_eval:.1f} pp, ITW {100*m_itw:.1f} pp, shrink {100*shrink:.1f} pp",
     "verdict": "edge shrinks (>= 10 pp)" if shrink >= 0.10 else "no shrinkage of >= 10 pp (not decidable as shrinking)"},
    {"hypothesis": "H4",
     "evidence": "ITW error at dev threshold minus ITW EER: " + ", ".join(f"rung {r}: {100*g:.1f} pp" for r, g in h4_gaps.items()),
     "verdict": "true" if (h4_gaps >= 0.10).sum() >= 2 else "false"},
])
verdicts.to_csv(ROOT / "results/e4_hypotheses.csv", index=False)
verdicts

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(figsize=(6.4, 3.4))
names = ["rung 0\nsilence/duration", "rung 1\nLFCC + gradient boosting", "rung 2\nfrozen XLS-R + LR"]
bars = [("eval_eer", "ASVspoof eval (EER)", "#52514e", None),
        ("itw_eer", "In-the-Wild (EER)", "#e34948", None),
        ("itw_err_at_dev_thr", "In-the-Wild at dev threshold", "#f2a3a2", "////")]
w = 0.26
for j, (col, lab, colr, hatch) in enumerate(bars):
    x = np.arange(3) + (j - 1) * w
    b = ax.bar(x, 100 * e4[col], width=w - 0.02, color=colr, hatch=hatch, edgecolor="white", label=lab)
    for xi, v in zip(x, 100 * e4[col]):
        ax.text(xi, v + 1, f"{v:.0f}", ha="center", fontsize=7.5, color=INK)
ax.axhline(50, color=MUTED, ls=":", lw=1); ax.text(-0.45, 51, "chance (50%)", fontsize=8, color=MUTED)
ax.set_xticks(range(3), names); ax.set_ylim(0, 90); ax.set_ylabel("error %")
ax.grid(axis="y", color=GRID, linewidth=0.6); ax.legend(frameon=False, fontsize=8, loc="upper right")
ax.set_title("Same models, new data source", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/e4_itw.png", dpi=200)
fig.savefig(ROOT / "results/figures/e4_itw.pdf")

---
# 6. E5: shortcuts, mitigation, calibration, failure analysis

Silence trimming and gain tests (H1b), augmentation retraining, ECE and temperature scaling, failures by duration, silence, gender and speaker. *(Source: `notebooks/05_shortcuts_calibration.ipynb`.)*

## 05 E5: shortcuts, mitigation (and later calibration)

All models are the final ones (chosen on dev, notebooks 02-03), refit from code with seed 0.
Test-time changes (`deepvoice.features.apply_transform`): **trim** (cut leading/trailing silence
with the group-S VAD), **gain-12** (12 dB quieter), **gain+6** (6 dB louder, clipped). Features of
changed audio are cached as `<group>_<split>_<change>.parquet`.

- **H1b** (frozen): trimming silence raises the rung 1 eval EER by >= 5 pp with the paired
  speaker-bootstrap CI excluding 0.
- XLS-R normalizes every clip to zero mean and unit variance, so gain changes cannot change its
  input (apart from clipped samples at +6 dB); it gets the trim test only.
- **Mitigation (E5.2):** rung 1 retrained on augmented train audio (trim, random 0-1 s noise padding,
  random -12..+6 dB gain; seeded per clip), same hyperparameters, compared on the same clips.

Results: `results/e5.csv` (standard columns; `split` names the change) and
`results/e5_diffs.csv` (paired differences with 95% CIs).

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.data import load_itw
from deepvoice.features import feature_tables, load_xy
from deepvoice.metrics import paired_speaker_bootstrap_diff
from deepvoice.models import run

# cached features of changed audio (computed once; loaded here)
p = pd.read_parquet("data/protocols.parquet")
sets = {"eval": p[p.split == "eval"].reset_index(drop=True), "itw": load_itw().reset_index(drop=True)}
for change in ["trim", "gain-12", "gain+6"]:
    for s, df in sets.items():
        feature_tables(df, f"{s}_{change}", transform=change)
feature_tables(p[p.split == "train"].reset_index(drop=True), "train_augment", transform="augment")

FS = {"groups": ("lfcc",), "n_coef": 40, "deltas": 2, "stats": ("mean", "std")}
gap = pd.read_csv(ROOT / "results/e3_gap.csv")
HGB = json.loads(gap[(gap.model == "hgb") & (gap["selected by"] == "dev")].params.iloc[0])
EV = {"dev": load_xy("dev", **FS)}
for s in ["eval", "itw"]:
    EV[s] = load_xy(s, **FS)
    for change in ["trim", "gain-12", "gain+6"]:
        EV[f"{s}_{change}"] = load_xy(s, variant=change, **FS)
train, train_aug = load_xy("train", **FS), load_xy("train", variant="augment", **FS)
X = list(train.columns[4:])
DATASET = lambda s: "itw" if s.startswith("itw") else "asvspoof"
print({k: len(v) for k, v in EV.items()})

## Rung 1: original vs augmented training, on clean and changed audio

In [ ]:
models, scores = {}, {}
for tag, tr in [("lfcc40dd_e3devsel", train), ("lfcc40dd_e3devsel_augment", train_aug)]:
    m, _ = run(1, "hgb", X, tr, [(DATASET(s), s, EV[s]) for s in EV], params=HGB,
               experiment="e5", feature_set=tag)
    models[tag] = m
    scores[tag] = {s: m.predict_proba(EV[s][X].to_numpy())[:, 1] for s in EV}
e5 = pd.read_csv(ROOT / "results/e5.csv")
e5[e5.rung == 1].assign(eer=lambda d: (100 * d.eer).round(2)).pivot_table(index="split", columns="model", values="eer")

## Paired differences (EER of B minus EER of A on the same clips, 95% CI)

In [ ]:
rows = []
orig, aug = scores["lfcc40dd_e3devsel"], scores["lfcc40dd_e3devsel_augment"]
for s in ["eval", "itw"]:
    for change in ["trim", "gain-12", "gain+6"]:  # same clips and labels, changed audio
        d, lo, hi = paired_speaker_bootstrap_diff(EV[s].label, orig[s], orig[f"{s}_{change}"], EV[s].speaker)
        rows.append({"model": "rung 1 (original)", "A": f"{s} clean", "B": f"{s} {change}", "clips": s,
                     "diff_pp": 100 * d, "ci_low_pp": 100 * lo, "ci_high_pp": 100 * hi})
    for v in ["", "_trim"]:
        d, lo, hi = paired_speaker_bootstrap_diff(EV[s].label, orig[s + v], aug[s + v], EV[s].speaker)
        rows.append({"model": "augmented minus original", "A": f"original on {s}{v.replace('_', ' ')}",
                     "B": f"augmented on {s}{v.replace('_', ' ')}", "clips": s,
                     "diff_pp": 100 * d, "ci_low_pp": 100 * lo, "ci_high_pp": 100 * hi})
diffs = pd.DataFrame(rows)
diffs.to_csv(ROOT / "results/e5_diffs.csv", index=False, float_format="%.3f")
diffs.round(2)

## H1b verdict (rule from the frozen `hypotheses.md`)

In [ ]:
h = diffs[(diffs.model == "rung 1 (original)") & (diffs.B == "eval trim")].iloc[0]
h1b = h.diff_pp >= 5 and h.ci_low_pp > 0
print(f"Trimming raises rung 1 eval EER by {h.diff_pp:.1f} pp [{h.ci_low_pp:.1f}, {h.ci_high_pp:.1f}] -> H1b {'TRUE' if h1b else 'FALSE'}")

## Rung 2 (frozen XLS-R layer 3 + logistic regression): clean vs trimmed

In [ ]:
from deepvoice.xlsr import LAYERS, extract_cached

extract_cached(sets["eval"], "eval_trim", transform="trim")   # cached: loads, never recomputes
extract_cached(sets["itw"], "itw_trim", transform="trim")
K, COLS = LAYERS.index(3), [f"x{i:04d}" for i in range(1024)]
META = {"train": train, "dev": EV["dev"], "eval": EV["eval"], "eval_trim": EV["eval"], "itw": EV["itw"], "itw_trim": EV["itw"]}


def xframe(name):
    meta = META[name]
    assert Path(f"data/features/xlsr300m_{name}_files.txt").read_text().split() == list(meta.file)
    e = np.asarray(np.load(f"data/features/xlsr300m_{name}.npy", mmap_mode="r")[:, K], dtype=np.float32)
    return pd.concat([meta[["file", "label", "speaker"]], pd.DataFrame(e, columns=COLS)], axis=1)


R2 = {s: xframe(s) for s in META}
m2, _ = run(2, "logreg", COLS, R2["train"], [(DATASET(s), s, R2[s]) for s in META if s != "train"],
            params={"C": 1.0}, experiment="e5", feature_set="xlsr300m_L03")
s2 = {s: m2.predict_proba(R2[s][COLS].to_numpy())[:, 1] for s in META if s != "train"}
for s in ["eval", "itw"]:
    d, lo, hi = paired_speaker_bootstrap_diff(EV[s].label, s2[s], s2[f"{s}_trim"], EV[s].speaker)
    rows.append({"model": "rung 2 (XLS-R)", "A": f"{s} clean", "B": f"{s} trim", "clips": s,
                 "diff_pp": 100 * d, "ci_low_pp": 100 * lo, "ci_high_pp": 100 * hi})
diffs = pd.DataFrame(rows)
diffs.to_csv(ROOT / "results/e5_diffs.csv", index=False, float_format="%.3f")
diffs[diffs.model == "rung 2 (XLS-R)"].round(2)

## Calibration (E5.3): reliability, ECE (15 bins), temperature scaling fit on dev

All three final models were trained with balanced class weights, so their probabilities assume a
50/50 world while the data is about 90% fake (ITW: 37%); temperature scaling (one number, no bias)
can soften over-confidence but cannot fix that prior shift.

In [ ]:
from deepvoice.metrics import apply_temperature, ece, fit_temperature, reliability
from deepvoice.models import build

SIL = ["lead_sil", "trail_sil", "sil_ratio", "duration"]
silf = {s: load_xy(s, groups=("silence",)) for s in ["train", "dev", "eval", "itw"]}
m0 = build("tree", 0, max_depth=3).fit(silf["train"][SIL].to_numpy(), silf["train"].label.to_numpy())
PROBS = {0: {s: m0.predict_proba(silf[s][SIL].to_numpy())[:, 1] for s in ["dev", "eval", "itw"]},
         1: {s: orig[s] for s in ["dev", "eval", "itw"]},
         2: {s: s2[s] for s in ["dev", "eval", "itw"]}}
cal = []
for rung, pr in PROBS.items():
    t = fit_temperature(EV["dev"].label, pr["dev"])
    for s in ["dev", "eval", "itw"]:
        y = EV[s].label
        cal.append({"rung": rung, "split": s, "temperature": t, "ece_before": ece(y, pr[s]),
                    "ece_after": ece(y, apply_temperature(pr[s], t))})
cal = pd.DataFrame(cal)
cal.to_csv(ROOT / "results/e5_calibration.csv", index=False, float_format="%.4f")
cal.assign(ece_before=lambda d: (100 * d.ece_before).round(1), ece_after=lambda d: (100 * d.ece_after).round(1))

## Failure analysis (E5.5): where do rungs 1 and 2 fail?

In [ ]:
from deepvoice.data import load_gender
from deepvoice.metrics import eer

gender = load_gender()
fail = []
def bucket_eers(rung, s, sc, factor, buckets):
    d = EV[s][["label"]].assign(score=sc, b=buckets)
    for b, g in d.groupby("b", observed=True):
        if g.label.nunique() == 2:
            fail.append({"rung": rung, "dataset": s, "factor": factor, "bucket": str(b),
                         "n": len(g), "eer": eer(g.label, g.score)[0]})

for rung, sc in [(1, orig), (2, s2)]:
    for s in ["eval", "itw"]:
        sf = silf[s]
        bucket_eers(rung, s, sc[s], "duration (s)", pd.cut(sf.duration, [0, 2, 3, 4, 6, 100]))
        bucket_eers(rung, s, sc[s], "silence ratio", pd.cut(sf.sil_ratio, [-0.01, 0.2, 0.35, 0.5, 1]))
    bucket_eers(rung, "eval", sc["eval"], "gender", EV["eval"].speaker.map(gender).fillna("unknown"))
fail = pd.DataFrame(fail)
fail.to_csv(ROOT / "results/e5_failures.csv", index=False, float_format="%.4f")
fail.assign(eer=lambda d: (100 * d.eer).round(1)).pivot_table(index=["dataset", "factor", "bucket"], columns="rung", values="eer")

In [ ]:
# per In-the-Wild speaker (rung 2): EER for each of the 54 speakers
spk = []
for name, g in EV["itw"].assign(score=s2["itw"]).groupby("speaker"):
    spk.append({"speaker": name, "n": len(g), "fake_share": g.label.mean(), "eer": eer(g.label, g.score)[0]})
spk = pd.DataFrame(spk).sort_values("eer")
spk.to_csv(ROOT / "results/e5_itw_speakers.csv", index=False, float_format="%.4f")
print("speakers with EER < 20%:", (spk.eer < 0.2).sum(), "| above 50% (worse than chance):", (spk.eer > 0.5).sum(), "of", len(spk))
pd.concat([spk.head(5), spk.tail(5)]).round(3)

In [ ]:
# the most confident mistakes of rung 2 on eval, at its dev threshold
thr2 = eer(EV["dev"].label, s2["dev"])[1]
ev = EV["eval"][["file", "label", "attack", "speaker"]].assign(score=s2["eval"], duration=silf["eval"].duration,
                                                                 lead_sil=silf["eval"].lead_sil)
fa = ev[(ev.label == 1) & (ev.score < thr2)].sort_values("score").head(200)   # fakes let through
fr = ev[(ev.label == 0) & (ev.score >= thr2)].sort_values("score", ascending=False).head(200)  # real flagged
print(f"dev threshold {thr2:.3f}: {len(ev[(ev.label == 1) & (ev.score < thr2)])} fakes accepted, {len(ev[(ev.label == 0) & (ev.score >= thr2)])} real clips rejected")
print("most confident false accepts, by attack:", fa.attack.value_counts().head(6).to_dict())
print("median duration / lead silence: false accepts", fa.duration.median().round(2), fa.lead_sil.median().round(2),
      "| false rejects", fr.duration.median().round(2), fr.lead_sil.median().round(2),
      "| all eval", ev.duration.median().round(2), ev.lead_sil.median().round(2))
print("false rejects: speakers", fr.speaker.nunique(), "| top:", fr.speaker.value_counts().head(5).to_dict())

## Figures

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
e5 = pd.read_csv(ROOT / "results/e5.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1.4, 1]})
conds = ["eval", "eval_trim", "eval_gain-12", "eval_gain+6", "itw", "itw_trim"]
labels = ["eval", "eval\ntrimmed", "eval\n-12 dB", "eval\n+6 dB", "ITW", "ITW\ntrimmed"]
series = [("hgb:lfcc40dd_e3devsel", "rung 1 (LFCC)", "#eda100", None),
          ("hgb:lfcc40dd_e3devsel_augment", "rung 1 + augmentation", "#eda100", "////"),
          ("logreg:xlsr300m_L03", "rung 2 (XLS-R)", "#1baf7a", None)]
w = 0.27
for j, (mname, lab, col, hatch) in enumerate(series):
    d = e5[e5.model == mname].set_index("split")
    x = np.arange(len(conds)) + (j - 1) * w
    vals = [100 * d.eer.get(c, np.nan) for c in conds]
    ax[0].bar(x, vals, width=w - 0.02, color=col, hatch=hatch, edgecolor="white", label=lab)
    for xi, v in zip(x, vals):
        if not np.isnan(v):
            ax[0].text(xi, v + 1, f"{v:.0f}", ha="center", fontsize=7, color=INK)
ax[0].axhline(50, color=MUTED, ls=":", lw=1)
ax[0].set_xticks(range(len(conds)), labels); ax[0].set_ylim(0, 95); ax[0].set_ylabel("EER %")
ax[0].grid(axis="y", color=GRID, linewidth=0.6); ax[0].legend(frameon=False, fontsize=8, loc="upper left")
ax[0].set_title("(a) Test-time changes and augmentation (XLS-R is gain-invariant: no gain bars)", loc="left", color=INK)

for s, col, lab in [("dev", "#c3c2b7", "ASVspoof dev"), ("eval", "#52514e", "ASVspoof eval"), ("itw", "#e34948", "In-the-Wild")]:
    r = reliability(EV[s].label, PROBS[2][s])
    e = cal[(cal.rung == 2) & (cal.split == s)].ece_before.iloc[0]
    ax[1].plot(r.mean_prob, r.frac_fake, marker="o", ms=4, lw=1.8, color=col, label=f"{lab} (ECE {100 * e:.0f}%)")
ax[1].plot([0, 1], [0, 1], color=MUTED, ls="--", lw=1)
ax[1].set_xlabel("predicted P(fake)"); ax[1].set_ylabel("observed fake rate")
ax[1].grid(color=GRID, linewidth=0.6); ax[1].legend(frameon=False, fontsize=8, loc="lower right")
ax[1].set_title("(b) Reliability of rung 2 (before scaling)", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/e5_shortcuts_calibration.png", dpi=200)
fig.savefig(ROOT / "results/figures/e5_shortcuts_calibration.pdf")

---
# 7. E1 cost table and the final model

Extraction, training and scoring cost per rung; saving and checking the final model. *(Source: `notebooks/06_cost_and_final_model.ipynb`.)*

## 06 E1 cost table and the final model

**Cost per rung**, measured the same way for all three final models: feature extraction per clip on
200 random eval clips (silence and classical features on one CPU core; XLS-R on the Apple M5 GPU,
`mps`), training time and scoring latency from the recorded runs (`results/rung*.csv`), and model
size (serialized bytes, trainable and frozen parameters). Output: `results/cost_table.csv`.

**Final model** = rung 2 (frozen XLS-R-300M layer 3 + logistic regression), the best model on ASVspoof
eval. Saved to `model/` with its metadata (layer, dev EER threshold, results, sha256); scored with
`python -m deepvoice.predict`. See `model/MODEL_CARD.md`.

In [ ]:
import hashlib
import io
import json
import os
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice import predict, xlsr
from deepvoice.features import clip_features, load_audio, load_xy
from deepvoice.metrics import eer
from deepvoice.models import build
from deepvoice.silence import silence_features

p = pd.read_parquet("data/protocols.parquet")
sample = list(p[p.split == "eval"].sample(200, random_state=0).path)

## Feature extraction time per clip

In [ ]:
def per_clip_ms(fn):
    fn(sample[0])  # warm-up
    t0 = time.perf_counter()
    for path in sample:
        fn(path)
    return (time.perf_counter() - t0) / len(sample) * 1000

ext = {0: per_clip_ms(lambda q: silence_features(load_audio(q))),
       1: per_clip_ms(clip_features)}  # computes MFCC + LFCC + spectral: an upper bound for LFCC alone
xm = xlsr.load_model()
ext[2] = xlsr.time_it(sample, xm)["s_per_clip"] * 1000
frozen_params = sum(t.numel() for t in xm.parameters())
{k: round(v, 1) for k, v in ext.items()}

## The three final models: refit (deterministic, seed 0), size, recorded train and scoring time

In [ ]:
gap = pd.read_csv("results/e3_gap.csv")
hgb_params = json.loads(gap[(gap.model == "hgb") & (gap["selected by"] == "dev")].params.iloc[0])
SIL = ["lead_sil", "trail_sil", "sil_ratio", "duration"]
sil = {s: load_xy(s, groups=("silence",)) for s in ["train", "dev"]}
lf = {s: load_xy(s, groups=("lfcc",), n_coef=40) for s in ["train", "dev"]}
LF = list(lf["train"].columns[4:])
K = xlsr.LAYERS.index(3)
xl = {s: np.asarray(np.load(f"data/features/xlsr300m_{s}.npy", mmap_mode="r")[:, K], dtype=np.float32) for s in ["train", "dev"]}
y = {s: sil[s].label.to_numpy() for s in ["train", "dev"]}

m0 = build("tree", 0, max_depth=3).fit(sil["train"][SIL].to_numpy(), y["train"])
m1 = build("hgb", 0, **hgb_params).fit(lf["train"][LF].to_numpy(), y["train"])
m2 = build("logreg", 0, C=1.0).fit(xl["train"], y["train"])

def nbytes(m):
    b = io.BytesIO(); joblib.dump(m, b); return b.tell()

trainable = {0: int(m0.tree_.node_count),
             1: int(sum(len(pr[0].nodes) for pr in m1._predictors)),
             2: int(m2[-1].coef_.size + m2[-1].intercept_.size)}
CONFIG = {0: ("rung0", "rung0_tree_s0"), 1: ("rung1", "rung1_hgb_lfcc40dd_e3devsel_s0"), 2: ("rung2", "rung2_logreg_xlsr300m_L03_s0")}
rows = []
for rung, (exp, cfg) in CONFIG.items():
    r = pd.read_csv(f"results/{exp}.csv").query("config == @cfg").set_index("split")
    rows.append({
        "rung": rung,
        "model": {0: "depth-3 tree, 4 silence/duration numbers", 1: "gradient boosting, LFCC-40 + deltas (240)",
                  2: "frozen XLS-R-300M layer 3 + logistic regression"}[rung],
        "extract_ms_per_clip": ext[rung], "extract_device": "mps (GPU)" if rung == 2 else "1 CPU core",
        "train_s": r.loc["dev", "train_time"], "score_ms_per_clip": r.loc["eval", "latency"],
        "model_kb": nbytes([m0, m1, m2][rung]) / 1024, "trainable_params_or_nodes": trainable[rung],
        "frozen_params": frozen_params if rung == 2 else 0,
        "eval_eer_pct": 100 * r.loc["eval", "eer"], "itw_eer_pct": 100 * r.loc["itw", "eer"]})
cost = pd.DataFrame(rows)
cost.to_csv("results/cost_table.csv", index=False, float_format="%.4g")
cost.round(3)

## Save the final model (rung 2) and check the scorer reproduces the pipeline

In [ ]:
s_dev = m2.predict_proba(xl["dev"])[:, 1]
dev_eer, thr = eer(y["dev"], s_dev)
Path("model").mkdir(exist_ok=True)
joblib.dump(m2, Path("model") / predict.MODEL_FILE)
sha = hashlib.sha256((Path("model") / predict.MODEL_FILE).read_bytes()).hexdigest()
r2 = pd.read_csv("results/rung2.csv").query("config == 'rung2_logreg_xlsr300m_L03_s0'").set_index("split")
meta = {"name": "DeepVoice rung 2", "xlsr_model": xlsr.MODEL, "layer": 3, "pooling": "mean over time",
        "classifier": "StandardScaler + LogisticRegression(C=1, class_weight=balanced)",
        "train_data": "ASVspoof 2019 LA train (25,380 clips, 16 kHz mono)", "seed": 0,
        "threshold": thr, "threshold_rule": "EER threshold on ASVspoof 2019 LA dev",
        "eer": {s: round(float(r2.loc[s, "eer"]), 6) for s in ["dev", "eval", "itw"]},
        "eer_ci95": {s: [round(float(r2.loc[s, "ci_low"]), 6), round(float(r2.loc[s, "ci_high"]), 6)] for s in ["eval", "itw"]},
        "sha256": sha}
(Path("model") / predict.MODEL_FILE.replace(".joblib", ".json")).write_text(json.dumps(meta, indent=2) + "\n")

# the CLI path (load file -> XLS-R -> layer 3 -> classifier) must give the pipeline's scores
check = p[p.split == "dev"].reset_index(drop=True).sample(8, random_state=1)
clf, meta_loaded = predict.load()
got = predict.score(check.path.tolist(), clf, meta_loaded, xm)
want = m2.predict_proba(xl["dev"][check.index])[:, 1]
print("max |difference| CLI vs pipeline:", float(np.max(np.abs(got - want))))
assert np.allclose(got, want, atol=0.02)
meta